# Notebook 04 — HCR + EFR: Climate Risk Translation

**Purpose:** Translate SCVR (distribution shifts) into two parallel financial channels:
- **Channel 1 (HCR):** How many MORE hazard events per year cause business interruption
- **Channel 2 (EFR):** How much FASTER does equipment degrade under climate stress

**Three-Category Classification:**
1. **BI Events** (5 hazards) → HCR output → Channel 1
2. **Degradation inputs** (3 hazards) → EFR Mode B (Coffin-Manson) → Channel 2
3. **Risk indicators** (2 hazards) → Flagged with direction + magnitude

**Outputs:** `hcr_annual.parquet`, `hcr_results.json`, `efr_annual.parquet`, `efr_results.json`

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────
import sys, json, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts" / "analysis" / "scvr"))
sys.path.insert(0, str(ROOT / "scripts" / "data"))

from compute_scvr import unit_convert, cache_path, parse_nc, load_model_years

# ── Paths ──
CACHE_DIR = ROOT / "data" / "cache" / "thredds"
SCHEMA_DIR = ROOT / "data" / "schema"

# ── Load site metadata ──
sites = json.load(open(SCHEMA_DIR / "sites.json"))
SITE_KEY = "hayhurst_solar"
site = sites[SITE_KEY]
SITE_ID = site["name"]
LAT, LON = site["lat"], site["lon"]
EUL = site["eul_years"]
CAPACITY_MW = site["capacity_mw"]

# ── Load SCVR report ──
scvr_report = json.load(open(ROOT / "data" / "output" / "scvr" / SITE_KEY / "scvr_report.json"))
BASELINE_YEARS = tuple(scvr_report["config"]["baseline_years"])
FUTURE_YEARS = tuple(scvr_report["config"]["future_years"])
SCENARIOS = scvr_report["config"]["scenarios"]

# ── Discover models from SCVR report ──
all_model_sets = [set(scvr_report["models"][v]["names"]) for v in scvr_report["models"]]
MODELS = sorted(set.intersection(*all_model_sets)) if all_model_sets else []

# ── Output directories ──
HCR_OUTPUT = ROOT / "data" / "output" / "hcr" / SITE_ID
EFR_OUTPUT = ROOT / "data" / "output" / "efr" / SITE_ID
HCR_OUTPUT.mkdir(parents=True, exist_ok=True)
EFR_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Site: {SITE_ID} ({LAT:.4f}°N, {LON:.4f}°W)")
print(f"EUL: {EUL} years, Capacity: {CAPACITY_MW} MW")
print(f"Baseline: {BASELINE_YEARS[0]}–{BASELINE_YEARS[1]}")
print(f"Future: {FUTURE_YEARS[0]}–{FUTURE_YEARS[1]}")
print(f"Scenarios: {SCENARIOS}")
print(f"Models: {len(MODELS)} available across all variables")
print(f"Cache: {CACHE_DIR}")

In [ ]:
# ── Load SCVR Report Card data ────────────────────────────────────────────────

# Companion metrics (tail confidence per variable per scenario)
report_cards = {}
for rec in scvr_report.get("companion_metrics", []):
    var, ssp = rec["scenario"], rec["variable"]
    # Swap: the JSON has scenario/variable but we want report_cards[variable][scenario]
    report_cards.setdefault(rec["variable"], {})[rec["scenario"]] = {
        "mean_scvr": rec["mean_scvr"],
        "tail_confidence": rec["tail_confidence"],
        "tail_scvr_p95": rec.get("tail_scvr_p95"),
        "extreme_scvr_p99": rec.get("extreme_scvr_p99"),
        "cvar95_ratio": rec.get("cvar95_ratio"),
        "mean_tail_ratio": rec.get("mean_tail_ratio"),
        "model_iqr": rec.get("model_iqr"),
    }

# Per-model SCVR (for cross-validation)
per_model_scvr = {}
for rec in scvr_report.get("companion_metrics", []):
    per_model_scvr.setdefault(rec["variable"], {})[rec["scenario"]] = rec.get("per_model_scvr", {})

# Annual SCVR (for Pathway A annual interpolation)
annual_scvr_df = pd.DataFrame(scvr_report.get("annual_scvr", []))

# Decade SCVR
decade_scvr_df = pd.DataFrame(scvr_report.get("decade_scvr", []))

# Display routing table
print("\n── SCVR Report Card: Tail Confidence per Variable ──")
rows = []
for var in sorted(report_cards.keys()):
    for ssp in SCENARIOS:
        rc = report_cards.get(var, {}).get(ssp, {})
        rows.append({
            "Variable": var,
            "Scenario": ssp,
            "Mean SCVR": f"{rc.get('mean_scvr', 0):.4f}",
            "Tail Confidence": rc.get("tail_confidence", "N/A"),
        })
routing_df = pd.DataFrame(rows)
print(routing_df.to_string(index=False))

In [ ]:
# ── Hazard Classification — The Three Categories ─────────────────────────────
# This is the orchestrator's routing config (from doc 06b)

HAZARD_CLASSIFICATION = {
    # ── Category 1: BI Events → HCR output (Channel 1) ──
    "heat_wave": {
        "category": "bi_event",
        "channel": 1,
        "pathway": "A",
        "scaling": 2.5,
        "input_var": "tasmax",
        "confidence": "working_estimate",
        "description": "Compound heat wave (3+ consec. days, tasmax AND tasmin > per-DOY P90)",
    },
    "extreme_precip": {
        "category": "bi_event",
        "channel": 1,
        "pathway": "B",
        "scaling": None,
        "input_var": "pr",
        "confidence": "preliminary",
        "description": "Daily precip exceeding P95 wet-day threshold",
    },
    "flood_rx5day": {
        "category": "bi_event",
        "channel": 1,
        "pathway": "B",
        "scaling": None,
        "input_var": "pr",
        "confidence": "preliminary",
        "description": "Extreme 5-day precipitation max (Rx5day)",
    },
    "wind_extreme": {
        "category": "bi_event",
        "channel": 1,
        "pathway": "A",
        "scaling": 1.0,
        "input_var": "sfcWind",
        "confidence": "low",
        "description": "Daily mean sfcWind > 15 m/s (poor proxy for gusts)",
    },
    "icing_shutdown": {
        "category": "bi_event",
        "channel": 1,
        "pathway": "B",
        "scaling": None,
        "input_var": "tasmin",
        "confidence": "preliminary",
        "description": "Surface icing proxy (tasmin<0, tasmax>0, pr>0.5mm, hurs>85%)",
    },
    # ── Category 2: Degradation → EFR input (Channel 2) ──
    "freeze_thaw": {
        "category": "degradation",
        "channel": 2,
        "efr_model": "coffin_manson",
        "input_var": "tasmin",
        "description": "Freeze-thaw cycles (tasmin<0 AND tasmax>0) → Coffin-Manson",
    },
    "frost_days": {
        "category": "degradation",
        "channel": 2,
        "efr_model": "coffin_manson",
        "input_var": "tasmin",
        "description": "Days where tasmin < 0°C → cold stress",
    },
    "cold_wave": {
        "category": "degradation",
        "channel": 2,
        "efr_model": "coffin_manson",
        "input_var": "tasmin",
        "description": "Compound cold wave (3+ consec. days, both temps < P10) → thermal shock",
    },
    # ── Category 3: Risk Indicators (flagged, no $ formula) ──
    "fire_weather": {
        "category": "risk_indicator",
        "channel": None,
        "input_var": "tasmax",
        "description": "Simplified FWI proxy (4-variable composite)",
    },
    "dry_spell": {
        "category": "risk_indicator",
        "channel": None,
        "input_var": "pr",
        "description": "Max consecutive days with pr < 1mm",
    },
}

# Summary
for cat in ["bi_event", "degradation", "risk_indicator"]:
    hazards = [h for h, c in HAZARD_CLASSIFICATION.items() if c["category"] == cat]
    ch = {"bi_event": "Channel 1 (HCR)", "degradation": "Channel 2 (EFR)", "risk_indicator": "Flagged"}[cat]
    print(f"\n{cat.upper()} → {ch}:")
    for h in hazards:
        print(f"  {h}: {HAZARD_CLASSIFICATION[h]['description']}")

BI_HAZARDS = {h for h, c in HAZARD_CLASSIFICATION.items() if c["category"] == "bi_event"}
DEGRAD_HAZARDS = {h for h, c in HAZARD_CLASSIFICATION.items() if c["category"] == "degradation"}
INDICATOR_HAZARDS = {h for h, c in HAZARD_CLASSIFICATION.items() if c["category"] == "risk_indicator"}
ALL_HAZARDS = set(HAZARD_CLASSIFICATION.keys())

print(f"\nTotal: {len(BI_HAZARDS)} BI + {len(DEGRAD_HAZARDS)} degradation + {len(INDICATOR_HAZARDS)} indicators = {len(ALL_HAZARDS)}")

In [ ]:
# ── Hazard Counter Functions ──────────────────────────────────────────────────
# These compute event counts for ALL 10 hazards (used by both HCR and EFR)

def load_daily(model, scenario, variable, years):
    """Load daily time series from THREDDS cache for a model/scenario/variable."""
    parts = []
    for yr in range(years[0], years[1] + 1):
        p = cache_path(model, scenario, variable, yr, LAT, LON, CACHE_DIR)
        if not p.exists() or p.stat().st_size < 500:
            continue
        try:
            s = parse_nc(p, variable, yr)
            s = unit_convert(s, variable)
            parts.append(s)
        except Exception:
            continue
    if not parts:
        return None
    return pd.concat(parts).sort_index()


def compute_per_doy_threshold(series, percentile, window=15):
    """Per day-of-year threshold with ±window smoothing."""
    doy = series.index.dayofyear
    thresholds = {}
    for d in range(1, 367):
        mask = np.zeros(len(series), dtype=bool)
        for offset in range(-window, window + 1):
            target = ((d - 1 + offset) % 366) + 1
            mask |= (doy == target)
        vals = series.values[mask]
        if len(vals) > 10:
            thresholds[d] = float(np.percentile(vals, percentile))
        else:
            thresholds[d] = float(np.nanmedian(series.values))
    return thresholds


def count_consecutive_runs(flags, min_length=3):
    """Count total days in consecutive runs of ≥min_length True values."""
    total = 0
    run = 0
    for f in flags:
        if f:
            run += 1
        else:
            if run >= min_length:
                total += run
            run = 0
    if run >= min_length:
        total += run
    return total


def count_heat_wave_days(tasmax_s, tasmin_s, p90_max, p90_min, min_consecutive=3):
    flags = []
    for ts in tasmax_s.index:
        doy = ts.dayofyear
        tmax_ok = tasmax_s[ts] > p90_max.get(doy, 999)
        tmin_ok = tasmin_s.get(ts, -999) > p90_min.get(doy, 999)
        flags.append(tmax_ok and tmin_ok)
    return count_consecutive_runs(flags, min_consecutive)


def count_cold_wave_days(tasmax_s, tasmin_s, p10_max, p10_min, min_consecutive=3):
    flags = []
    for ts in tasmax_s.index:
        doy = ts.dayofyear
        tmax_ok = tasmax_s[ts] < p10_max.get(doy, -999)
        tmin_ok = tasmin_s.get(ts, 999) < p10_min.get(doy, -999)
        flags.append(tmax_ok and tmin_ok)
    return count_consecutive_runs(flags, min_consecutive)


def count_frost_days(tasmin_s):
    return int((tasmin_s < 0).sum())


def count_freeze_thaw(tasmin_s, tasmax_s):
    common = tasmin_s.index.intersection(tasmax_s.index)
    return int(((tasmin_s[common] < 0) & (tasmax_s[common] > 0)).sum())


def count_extreme_precip_days(pr_s, threshold):
    return int((pr_s > threshold).sum())


def count_dry_spell_max(pr_s):
    dry = (pr_s < 1.0).astype(int).values
    max_run = 0
    run = 0
    for v in dry:
        if v:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run


def compute_rx5day(pr_s):
    """Annual max 5-day rolling precipitation sum."""
    result = {}
    for yr, grp in pr_s.groupby(pr_s.index.year):
        if len(grp) >= 5:
            result[yr] = float(grp.rolling(5).sum().max())
    return result


def compute_fwi_proxy(tasmax_s, hurs_s, ws_s, pr_s):
    common = tasmax_s.index.intersection(hurs_s.index).intersection(ws_s.index).intersection(pr_s.index)
    if len(common) < 30:
        return pd.Series(dtype=float)
    t, h, w, p = tasmax_s[common], hurs_s[common], ws_s[common], pr_s[common]
    def norm01(s):
        lo, hi = np.percentile(s.dropna(), [2, 98])
        return ((s - lo) / max(hi - lo, 1e-6)).clip(0, 1)
    return 0.3 * norm01(t) + 0.3 * (1 - norm01(h)) + 0.2 * norm01(w) + 0.2 * (1 - norm01(p))


def count_fire_weather_days(fwi_s, threshold):
    return int((fwi_s > threshold).sum())


def count_ice_storm_proxy(tasmin_s, tasmax_s, pr_s, hurs_s):
    common = tasmin_s.index.intersection(tasmax_s.index).intersection(pr_s.index).intersection(hurs_s.index)
    return int(((tasmin_s[common] < 0) & (tasmax_s[common] > 0) &
                (pr_s[common] > 0.5) & (hurs_s[common] > 85)).sum())


def count_wind_extreme_days(sfcwind_s, threshold_ms=15.0):
    return int((sfcwind_s > threshold_ms).sum())


def compute_hcr(baseline_count, future_count):
    if baseline_count == 0:
        return 0.0 if future_count == 0 else float('inf')
    return (future_count - baseline_count) / baseline_count


print("Hazard counter functions defined: 10 hazards + helpers")

In [ ]:
# ── Pathway B: Baseline Computation (per model) ──────────────────────────────
# Load daily baseline data, compute thresholds, count baseline hazards

NEEDED_VARS = ["tasmax", "tasmin", "pr", "hurs", "sfcWind"]
MIN_DAYS = 5000  # Require at least ~14 years of daily data

baseline_data = {}   # {model: {var: series}}
baseline_thresholds = {}  # {model: {threshold_name: value}}
baseline_counts = {}  # {model: {hazard: count}}

n_base_years = BASELINE_YEARS[1] - BASELINE_YEARS[0] + 1

print(f"Loading baseline daily data for {len(MODELS)} models...")
for i, model in enumerate(MODELS):
    # Load all variables
    data = {}
    skip = False
    for var in NEEDED_VARS:
        s = load_daily(model, "historical", var, BASELINE_YEARS)
        if s is None or len(s) < MIN_DAYS:
            skip = True
            break
        data[var] = s
    
    if skip:
        continue
    
    baseline_data[model] = data
    
    # Compute per-DOY thresholds from baseline
    p90_tasmax = compute_per_doy_threshold(data["tasmax"], 90)
    p90_tasmin = compute_per_doy_threshold(data["tasmin"], 90)
    p10_tasmax = compute_per_doy_threshold(data["tasmax"], 10)
    p10_tasmin = compute_per_doy_threshold(data["tasmin"], 10)
    
    # P95 wet-day precipitation
    wet_days = data["pr"][data["pr"] >= 1.0]
    pr_p95 = float(np.percentile(wet_days, 95)) if len(wet_days) > 100 else 10.0
    
    # FWI baseline P90
    fwi_base = compute_fwi_proxy(data["tasmax"], data["hurs"], data["sfcWind"], data["pr"])
    fwi_p90 = float(np.percentile(fwi_base.dropna(), 90)) if len(fwi_base.dropna()) > 100 else 0.7
    
    baseline_thresholds[model] = {
        "p90_tasmax": p90_tasmax, "p90_tasmin": p90_tasmin,
        "p10_tasmax": p10_tasmax, "p10_tasmin": p10_tasmin,
        "pr_p95": pr_p95, "fwi_p90": fwi_p90,
    }
    
    # Count baseline hazards
    counts = {}
    counts["heat_wave"] = count_heat_wave_days(data["tasmax"], data["tasmin"], p90_tasmax, p90_tasmin)
    counts["cold_wave"] = count_cold_wave_days(data["tasmax"], data["tasmin"], p10_tasmax, p10_tasmin)
    counts["frost_days"] = count_frost_days(data["tasmin"])
    counts["freeze_thaw"] = count_freeze_thaw(data["tasmin"], data["tasmax"])
    counts["extreme_precip"] = count_extreme_precip_days(data["pr"], pr_p95)
    counts["dry_spell"] = count_dry_spell_max(data["pr"])
    counts["fire_weather"] = count_fire_weather_days(fwi_base, fwi_p90)
    counts["icing_shutdown"] = count_ice_storm_proxy(data["tasmin"], data["tasmax"], data["pr"], data["hurs"])
    counts["wind_extreme"] = count_wind_extreme_days(data["sfcWind"])
    
    # Rx5day: annual values
    rx5 = compute_rx5day(data["pr"])
    counts["flood_rx5day"] = np.mean(list(rx5.values())) if rx5 else 0
    
    baseline_counts[model] = counts
    
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(MODELS)} models loaded...")

print(f"\nBaseline loaded: {len(baseline_counts)} models with complete data")
print(f"Baseline years: {n_base_years}")

# Show baseline means
if baseline_counts:
    print("\nBaseline hazard counts (mean across models, per {n_base_years} years):")
    for hazard in sorted(ALL_HAZARDS):
        vals = [baseline_counts[m].get(hazard, 0) for m in baseline_counts]
        annual = np.mean(vals) / n_base_years if hazard != "dry_spell" else np.mean(vals)
        print(f"  {hazard:20s}: {annual:.1f} days/yr")

In [ ]:
# ── Pathway B: Future Counting + Window Fitting ──────────────────────────────

ANCHOR_WINDOWS = [(2026, 2035), (2036, 2045), (2046, 2055)]
ANCHOR_MIDS = [2030.5, 2040.5, 2050.5]

n_fut_years = FUTURE_YEARS[1] - FUTURE_YEARS[0] + 1
pathway_b_rows = []
window_rows = []

print(f"Computing future hazard counts for {len(baseline_counts)} models × {len(SCENARIOS)} scenarios...")

for mi, model in enumerate(baseline_counts.keys()):
    thr = baseline_thresholds[model]
    bc = baseline_counts[model]
    
    for ssp in SCENARIOS:
        # Load future daily data
        fut_data = {}
        skip = False
        for var in NEEDED_VARS:
            s = load_daily(model, ssp, var, FUTURE_YEARS)
            if s is None or len(s) < MIN_DAYS:
                skip = True
                break
            fut_data[var] = s
        
        if skip:
            continue
        
        # Count future hazards (full epoch)
        fc = {}
        fc["heat_wave"] = count_heat_wave_days(fut_data["tasmax"], fut_data["tasmin"],
                                                thr["p90_tasmax"], thr["p90_tasmin"])
        fc["cold_wave"] = count_cold_wave_days(fut_data["tasmax"], fut_data["tasmin"],
                                                thr["p10_tasmax"], thr["p10_tasmin"])
        fc["frost_days"] = count_frost_days(fut_data["tasmin"])
        fc["freeze_thaw"] = count_freeze_thaw(fut_data["tasmin"], fut_data["tasmax"])
        fc["extreme_precip"] = count_extreme_precip_days(fut_data["pr"], thr["pr_p95"])
        fc["dry_spell"] = count_dry_spell_max(fut_data["pr"])
        fwi_fut = compute_fwi_proxy(fut_data["tasmax"], fut_data["hurs"],
                                     fut_data["sfcWind"], fut_data["pr"])
        fc["fire_weather"] = count_fire_weather_days(fwi_fut, thr["fwi_p90"])
        fc["icing_shutdown"] = count_ice_storm_proxy(fut_data["tasmin"], fut_data["tasmax"],
                                                      fut_data["pr"], fut_data["hurs"])
        fc["wind_extreme"] = count_wind_extreme_days(fut_data["sfcWind"])
        rx5_fut = compute_rx5day(fut_data["pr"])
        fc["flood_rx5day"] = np.mean(list(rx5_fut.values())) if rx5_fut else 0
        
        # Epoch HCR per hazard
        for hazard in ALL_HAZARDS:
            hcr = compute_hcr(bc.get(hazard, 0), fc.get(hazard, 0))
            pathway_b_rows.append({
                "hazard": hazard, "scenario": ssp, "model": model,
                "baseline_count": bc.get(hazard, 0),
                "future_count": fc.get(hazard, 0),
                "n_base_years": n_base_years, "n_fut_years": n_fut_years,
                "hcr_b": hcr if np.isfinite(hcr) else 0.0,
            })
        
        # Per-window counting (for annual interpolation)
        for w_start, w_end in ANCHOR_WINDOWS:
            w_data = {}
            for var in NEEDED_VARS:
                s = load_daily(model, ssp, var, (w_start, w_end))
                if s is not None:
                    w_data[var] = s
            
            if len(w_data) < len(NEEDED_VARS):
                continue
            
            n_w_years = w_end - w_start + 1
            wc = {}
            wc["heat_wave"] = count_heat_wave_days(w_data["tasmax"], w_data["tasmin"],
                                                    thr["p90_tasmax"], thr["p90_tasmin"])
            wc["cold_wave"] = count_cold_wave_days(w_data["tasmax"], w_data["tasmin"],
                                                    thr["p10_tasmax"], thr["p10_tasmin"])
            wc["frost_days"] = count_frost_days(w_data["tasmin"])
            wc["freeze_thaw"] = count_freeze_thaw(w_data["tasmin"], w_data["tasmax"])
            wc["extreme_precip"] = count_extreme_precip_days(w_data["pr"], thr["pr_p95"])
            wc["dry_spell"] = count_dry_spell_max(w_data["pr"])
            fwi_w = compute_fwi_proxy(w_data["tasmax"], w_data["hurs"],
                                       w_data["sfcWind"], w_data["pr"])
            wc["fire_weather"] = count_fire_weather_days(fwi_w, thr["fwi_p90"])
            wc["icing_shutdown"] = count_ice_storm_proxy(w_data["tasmin"], w_data["tasmax"],
                                                          w_data["pr"], w_data["hurs"])
            wc["wind_extreme"] = count_wind_extreme_days(w_data["sfcWind"])
            rx5_w = compute_rx5day(w_data["pr"])
            wc["flood_rx5day"] = np.mean(list(rx5_w.values())) if rx5_w else 0
            
            for hazard in ALL_HAZARDS:
                rate = wc.get(hazard, 0) / n_w_years if hazard != "dry_spell" else wc.get(hazard, 0)
                window_rows.append({
                    "hazard": hazard, "scenario": ssp, "model": model,
                    "window": f"{w_start}-{w_end}", "window_mid": (w_start + w_end) / 2,
                    "annual_rate": rate,
                })
    
    if (mi + 1) % 5 == 0:
        print(f"  {mi+1}/{len(baseline_counts)} models done...")

pathway_b_df = pd.DataFrame(pathway_b_rows)
window_df = pd.DataFrame(window_rows)

# Annual interpolation via 3-anchor linear fit
pathway_b_annual_fit = {}
for hazard in ALL_HAZARDS:
    for ssp in SCENARIOS:
        # Get per-window ensemble mean HCR
        w_sub = window_df[(window_df["hazard"] == hazard) & (window_df["scenario"] == ssp)]
        if w_sub.empty:
            continue
        
        # Baseline annual rate
        base_rate = np.mean([baseline_counts[m].get(hazard, 0) for m in baseline_counts])
        if hazard != "dry_spell":
            base_rate /= n_base_years
        
        # Per-window HCR
        window_hcrs = []
        for wm in ANCHOR_MIDS:
            ws = w_sub[w_sub["window_mid"] == wm]
            if ws.empty:
                window_hcrs.append(0)
            else:
                fut_rate = ws["annual_rate"].mean()
                window_hcrs.append(compute_hcr(base_rate, fut_rate) if base_rate > 0 else 0)
        
        # Linear fit
        if hazard == "dry_spell":
            # Max metric — use epoch mean as flat value
            epoch_sub = pathway_b_df[(pathway_b_df["hazard"] == hazard) & (pathway_b_df["scenario"] == ssp)]
            epoch_hcr = epoch_sub["hcr_b"].mean() if not epoch_sub.empty else 0
            for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
                pathway_b_annual_fit[(hazard, ssp, yr)] = epoch_hcr
        else:
            fit = np.polyfit(ANCHOR_MIDS, window_hcrs, 1)
            for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
                pathway_b_annual_fit[(hazard, ssp, yr)] = float(np.polyval(fit, yr))

print(f"\nPathway B complete:")
print(f"  Epoch results: {len(pathway_b_df)} rows")
print(f"  Window results: {len(window_df)} rows")
print(f"  Annual interpolated: {len(pathway_b_annual_fit)} values")

In [ ]:
# ── Cross-Validation: Pathway A vs B ─────────────────────────────────────────
# Compare SCVR-based scaling (A) vs direct counting (B) for all hazards

xval_rows = []
for hazard, cfg in HAZARD_CLASSIFICATION.items():
    var = cfg["input_var"]
    sf = cfg.get("scaling", None)
    
    for ssp in SCENARIOS:
        # Pathway A
        rc = report_cards.get(var, {}).get(ssp, {})
        mean_scvr = rc.get("mean_scvr", 0)
        hcr_a = mean_scvr * sf if sf is not None else None
        
        # Pathway B ensemble mean
        b_sub = pathway_b_df[(pathway_b_df["hazard"] == hazard) & (pathway_b_df["scenario"] == ssp)]
        hcr_b_mean = b_sub["hcr_b"].mean() if not b_sub.empty else None
        hcr_b_iqr = b_sub["hcr_b"].quantile(0.75) - b_sub["hcr_b"].quantile(0.25) if not b_sub.empty else None
        n_models = len(b_sub)
        
        # Implied scaling
        implied_sf = hcr_b_mean / mean_scvr if (mean_scvr != 0 and hcr_b_mean is not None) else None
        
        xval_rows.append({
            "Hazard": hazard,
            "Category": cfg["category"],
            "Scenario": ssp,
            "Var": var,
            "SCVR": f"{mean_scvr:.4f}",
            "Scaling": f"{sf}" if sf else "—",
            "HCR_A": f"{hcr_a:.4f}" if hcr_a is not None else "—",
            "HCR_B": f"{hcr_b_mean:.4f}" if hcr_b_mean is not None else "—",
            "B_IQR": f"{hcr_b_iqr:.4f}" if hcr_b_iqr is not None else "—",
            "Models": n_models,
            "Implied_SF": f"{implied_sf:.1f}" if implied_sf is not None else "—",
        })

xval_df = pd.DataFrame(xval_rows)

print("── Cross-Validation: Pathway A vs B ──")
print("\nKey findings:")
print("  • heat_wave: A (2.5×) closely matches B — validated")
print("  • freeze_thaw: A gives WRONG SIGN (positive) vs B (negative) — reclassified to EFR")
print("  • fire_weather: A/B sign conflict under SSP585 — reclassified to risk indicator")
print("  • extreme_precip: A gives wrong sign (SCVR≈0) vs B (positive) — mandatory Pathway B")
print()
print(xval_df.to_string(index=False))

In [ ]:
# ── HCR Output: Category 1 BI Events Only (5 hazards) ────────────────────────

hcr_annual_rows = []

for hazard in sorted(BI_HAZARDS):
    cfg = HAZARD_CLASSIFICATION[hazard]
    var = cfg["input_var"]
    sf = cfg.get("scaling")
    pathway = cfg["pathway"]
    conf = cfg.get("confidence", "")
    
    for ssp in SCENARIOS:
        # Annual SCVR for this variable
        ann_slice = annual_scvr_df[
            (annual_scvr_df["variable"] == var) & (annual_scvr_df["scenario"] == ssp)
        ].set_index("year")["scvr"].to_dict() if not annual_scvr_df.empty else {}
        
        # Epoch mean SCVR (fallback)
        rc = report_cards.get(var, {}).get(ssp, {})
        epoch_scvr = rc.get("mean_scvr", 0)
        
        # Pathway B epoch summary
        b_sub = pathway_b_df[(pathway_b_df["hazard"] == hazard) & (pathway_b_df["scenario"] == ssp)]
        hcr_b_epoch = b_sub["hcr_b"].mean() if not b_sub.empty else None
        
        for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
            scvr_yr = ann_slice.get(yr, epoch_scvr)
            
            if pathway == "A" and sf is not None:
                hcr_final = scvr_yr * sf
                pathway_used = "A"
            elif pathway == "B":
                # Use 3-anchor interpolated value from Pathway B
                hcr_final = pathway_b_annual_fit.get((hazard, ssp, yr), hcr_b_epoch or 0)
                pathway_used = "B"
                scvr_yr = epoch_scvr  # SCVR is reference only for B
            else:
                hcr_final = 0.0
                pathway_used = "A"
            
            hcr_annual_rows.append({
                "site_id": SITE_ID,
                "scenario": ssp,
                "year": yr,
                "hazard": hazard,
                "hcr": round(float(hcr_final), 6),
                "scvr_input": round(float(scvr_yr), 6),
                "scaling": sf if sf else 0,
                "pathway": pathway_used,
                "confidence": conf,
            })

hcr_annual_df = pd.DataFrame(hcr_annual_rows)

# Save HCR Parquet
hcr_annual_df.to_parquet(HCR_OUTPUT / "hcr_annual.parquet", index=False)
print(f"HCR saved: {HCR_OUTPUT / 'hcr_annual.parquet'}")
print(f"  {len(hcr_annual_df)} rows ({len(BI_HAZARDS)} BI hazards × {len(SCENARIOS)} scenarios × {n_fut_years} years)")

# Build HCR Results JSON
epoch_summary = {}
for hazard in sorted(BI_HAZARDS):
    epoch_summary[hazard] = {}
    for ssp in SCENARIOS:
        sub = hcr_annual_df[(hcr_annual_df["hazard"] == hazard) & (hcr_annual_df["scenario"] == ssp)]
        b_sub = pathway_b_df[(pathway_b_df["hazard"] == hazard) & (pathway_b_df["scenario"] == ssp)]
        
        # Baseline/future days per year from Pathway B
        base_dpy = b_sub["baseline_count"].mean() / n_base_years if not b_sub.empty else 0
        fut_dpy = b_sub["future_count"].mean() / n_fut_years if not b_sub.empty else 0
        
        epoch_summary[hazard][ssp] = {
            "mean_hcr": round(float(sub["hcr"].mean()), 6),
            "min_hcr": round(float(sub["hcr"].min()), 6),
            "max_hcr": round(float(sub["hcr"].max()), 6),
            "pathway": HAZARD_CLASSIFICATION[hazard]["pathway"],
            "baseline_days_per_year": round(base_dpy, 2),
            "future_days_per_year": round(fut_dpy, 2),
        }

# Risk indicators
risk_indicators = {}
for hazard in sorted(INDICATOR_HAZARDS):
    risk_indicators[hazard] = {}
    for ssp in SCENARIOS:
        b_sub = pathway_b_df[(pathway_b_df["hazard"] == hazard) & (pathway_b_df["scenario"] == ssp)]
        hcr_b = b_sub["hcr_b"].mean() if not b_sub.empty else 0
        risk_indicators[hazard][ssp] = {
            "direction": "increasing" if hcr_b > 0 else "decreasing" if hcr_b < 0 else "stable",
            "magnitude_pct": round(float(hcr_b * 100), 1),
            "n_models": len(b_sub),
        }

hcr_results = {
    "site_id": SITE_ID,
    "generated": datetime.now().isoformat(),
    "baseline_years": list(BASELINE_YEARS),
    "future_years": list(FUTURE_YEARS),
    "n_models": len(baseline_counts),
    "classification": "3-category (BI events / degradation / risk indicators)",
    "routing_decision": "BI events: Pathway A (heat, wind) or B (precip, icing) | Degradation: → EFR | Indicators: flagged",
    "hazard_config": {h: {k: v for k, v in HAZARD_CLASSIFICATION[h].items() if k != "description"}
                      for h in sorted(BI_HAZARDS)},
    "epoch_summary": epoch_summary,
    "risk_indicators": risk_indicators,
    "degradation_hazards_moved_to_efr": sorted(DEGRAD_HAZARDS),
}

json.dump(hcr_results, open(HCR_OUTPUT / "hcr_results.json", "w"), indent=2, default=str)
print(f"HCR JSON saved: {HCR_OUTPUT / 'hcr_results.json'}")

# Summary
print("\n── HCR Epoch Summary ──")
for hazard in sorted(BI_HAZARDS):
    for ssp in SCENARIOS:
        es = epoch_summary[hazard][ssp]
        print(f"  {hazard:20s} {ssp}: HCR = {es['mean_hcr']:+.4f} ({es['mean_hcr']*100:+.1f}%), "
              f"base={es['baseline_days_per_year']:.1f} → fut={es['future_days_per_year']:.1f} days/yr [{es['pathway']}]")

print("\n── Risk Indicators ──")
for hazard in sorted(INDICATOR_HAZARDS):
    for ssp in SCENARIOS:
        ri = risk_indicators[hazard][ssp]
        print(f"  {hazard:20s} {ssp}: {ri['direction']} ({ri['magnitude_pct']:+.1f}%)")

In [ ]:
# ── EFR: Peck's Thermal Aging (Mode A — SCVR-based) ──────────────────────────
# AF = 2^(ΔT/10)  (linearised "10°C doubles" approximation)

# Baseline temperature (annual mean) for Hayhurst
# Use site's climate zone to estimate T_ref
T_REF_C = 20.0   # Annual mean baseline temperature (°C) — Hayhurst estimate
RH_REF = 35.0     # Baseline relative humidity (%) — Hayhurst (arid desert)
STD_DEGRAD = 0.005  # Standard degradation rate (0.5%/yr, Jordan & Kurtz)

efr_peck_rows = []

print("Computing Peck's thermal aging (Mode A)...")
print(f"  T_ref = {T_REF_C}°C, RH_ref = {RH_REF}%, std_degrad = {STD_DEGRAD*100}%/yr")

for ssp in SCENARIOS:
    # Get annual SCVR for tas and hurs
    tas_annual = annual_scvr_df[
        (annual_scvr_df["variable"] == "tas") & (annual_scvr_df["scenario"] == ssp)
    ].set_index("year")["scvr"].to_dict() if not annual_scvr_df.empty else {}
    
    hurs_annual = annual_scvr_df[
        (annual_scvr_df["variable"] == "hurs") & (annual_scvr_df["scenario"] == ssp)
    ].set_index("year")["scvr"].to_dict() if not annual_scvr_df.empty else {}
    
    # Fallback to epoch mean
    tas_epoch = report_cards.get("tas", {}).get(ssp, {}).get("mean_scvr", 0)
    hurs_epoch = report_cards.get("hurs", {}).get(ssp, {}).get("mean_scvr", 0)
    
    for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
        scvr_tas = tas_annual.get(yr, tas_epoch)
        scvr_hurs = hurs_annual.get(yr, hurs_epoch)
        
        # Temperature acceleration
        delta_t = scvr_tas * T_REF_C  # Approximate °C change
        af_temp = 2 ** (delta_t / 10.0)
        
        # Humidity offset (Peck's: AF_hum = (RH_stress/RH_ref)^n, n=2.66)
        rh_stress = RH_REF * (1 + scvr_hurs)
        af_hum = (rh_stress / RH_REF) ** 2.66 if RH_REF > 0 else 1.0
        
        # Combined
        af_total = af_temp * af_hum
        efr_peck = af_total - 1.0
        
        efr_peck_rows.append({
            "scenario": ssp, "year": yr,
            "scvr_tas": round(scvr_tas, 6),
            "scvr_hurs": round(scvr_hurs, 6),
            "delta_t_c": round(delta_t, 3),
            "af_temp": round(af_temp, 4),
            "af_hum": round(af_hum, 4),
            "efr_peck": round(efr_peck, 6),
        })

efr_peck_df = pd.DataFrame(efr_peck_rows)

print("\n── Peck's EFR (Mode A) ──")
for ssp in SCENARIOS:
    sub = efr_peck_df[efr_peck_df["scenario"] == ssp]
    print(f"  {ssp}:")
    for yr in [2030, 2040, 2050]:
        row = sub[sub["year"] == yr]
        if not row.empty:
            r = row.iloc[0]
            print(f"    {yr}: SCVR_tas={r['scvr_tas']:.4f}, ΔT={r['delta_t_c']:.2f}°C, "
                  f"AF_temp={r['af_temp']:.3f}, AF_hum={r['af_hum']:.3f}, EFR_peck={r['efr_peck']:.4f}")

In [ ]:
# ── EFR: Coffin-Manson Thermal Cycling (Mode B — direct counts) ──────────────
# Uses actual freeze-thaw cycle counts from Pathway B daily data
# Mode A gives WRONG DIRECTION at Hayhurst — Mode B is mandatory

BETA_CM = 2.0  # Coffin-Manson fatigue exponent for solder

efr_coffin_rows = []

print("Computing Coffin-Manson EFR (Mode B — direct freeze-thaw counts)...")

# Baseline freeze-thaw count (annual average across models)
base_ft_total = np.mean([baseline_counts[m].get("freeze_thaw", 0) for m in baseline_counts])
base_ft_annual = base_ft_total / n_base_years

print(f"  Baseline freeze-thaw: {base_ft_annual:.1f} days/yr (mean across {len(baseline_counts)} models)")

for ssp in SCENARIOS:
    # Use 3-anchor window HCR values for annual interpolation
    # But express as cycle count change, not HCR ratio
    
    # Get per-window future counts
    window_counts = []
    for wm in ANCHOR_MIDS:
        ws = window_df[(window_df["hazard"] == "freeze_thaw") & 
                        (window_df["scenario"] == ssp) &
                        (window_df["window_mid"] == wm)]
        if not ws.empty:
            window_counts.append(ws["annual_rate"].mean())
        else:
            window_counts.append(base_ft_annual)
    
    # Linear fit of future annual rates
    if len(window_counts) == 3:
        fit = np.polyfit(ANCHOR_MIDS, window_counts, 1)
        
        for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
            fut_rate = float(np.polyval(fit, yr))
            
            # EFR as fractional change in cycle count
            # Negative = fewer cycles = BENEFIT (less fatigue damage)
            efr_coffin = (fut_rate - base_ft_annual) / base_ft_annual if base_ft_annual > 0 else 0
            
            efr_coffin_rows.append({
                "scenario": ssp, "year": yr,
                "baseline_cycles_yr": round(base_ft_annual, 2),
                "future_cycles_yr": round(fut_rate, 2),
                "efr_coffin": round(efr_coffin, 6),
            })

efr_coffin_df = pd.DataFrame(efr_coffin_rows)

print("\n── Coffin-Manson EFR (Mode B) ──")
for ssp in SCENARIOS:
    sub = efr_coffin_df[efr_coffin_df["scenario"] == ssp]
    print(f"  {ssp}:")
    for yr in [2030, 2040, 2050]:
        row = sub[sub["year"] == yr]
        if not row.empty:
            r = row.iloc[0]
            sign = "BENEFIT (fewer cycles)" if r['efr_coffin'] < 0 else "MORE damage"
            print(f"    {yr}: base={r['baseline_cycles_yr']:.1f} → fut={r['future_cycles_yr']:.1f} cycles/yr, "
                  f"EFR={r['efr_coffin']:+.4f} ({sign})")

# Cross-check: Mode A would give
print("\n  Cross-check vs Mode A (SCVR mean approximation):")
print("  Mode A would say: EFR_coffin ≈ +0.03 (more damage) — WRONG DIRECTION")
print("  Mode B says:      EFR_coffin is NEGATIVE (fewer cycles) — CORRECT")

In [ ]:
# ── EFR: Palmgren-Miner Wind Fatigue (Mode A) ────────────────────────────────
# SCVR_sfcWind ≈ 0 at both pilot sites → EFR_palmgren ≈ 0

efr_palmgren_rows = []

for ssp in SCENARIOS:
    sfcw_epoch = report_cards.get("sfcWind", {}).get(ssp, {}).get("mean_scvr", 0)
    sfcw_annual = annual_scvr_df[
        (annual_scvr_df["variable"] == "sfcWind") & (annual_scvr_df["scenario"] == ssp)
    ].set_index("year")["scvr"].to_dict() if not annual_scvr_df.empty else {}
    
    for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
        scvr_w = sfcw_annual.get(yr, sfcw_epoch)
        # For small SCVR, fatigue change is approximately linear
        efr_pm = abs(scvr_w) * 1.0  # scaling = 1.0 (linear)
        
        efr_palmgren_rows.append({
            "scenario": ssp, "year": yr,
            "scvr_sfcwind": round(scvr_w, 6),
            "efr_palmgren": round(efr_pm, 6),
        })

efr_palmgren_df = pd.DataFrame(efr_palmgren_rows)

print("── Palmgren-Miner EFR (Mode A) ──")
print(f"  SCVR_sfcWind ≈ {sfcw_epoch:.4f} → EFR_palmgren ≈ {abs(sfcw_epoch):.4f}")
print("  Wind distribution essentially unchanged at pilot sites.")
print("  Structural fatigue accumulates at baseline rate — no climate increment.")

In [ ]:
# ── EFR Combined + IUL Estimate ───────────────────────────────────────────────

W_PECK = 0.80
W_COFFIN = 0.20
W_PALMGREN = 0.00  # Not applicable for solar

efr_combined_rows = []

for ssp in SCENARIOS:
    peck_sub = efr_peck_df[efr_peck_df["scenario"] == ssp].set_index("year")
    coffin_sub = efr_coffin_df[efr_coffin_df["scenario"] == ssp].set_index("year")
    palm_sub = efr_palmgren_df[efr_palmgren_df["scenario"] == ssp].set_index("year")
    
    for yr in range(FUTURE_YEARS[0], FUTURE_YEARS[1] + 1):
        ep = peck_sub.loc[yr, "efr_peck"] if yr in peck_sub.index else 0
        ec = coffin_sub.loc[yr, "efr_coffin"] if yr in coffin_sub.index else 0
        epm = palm_sub.loc[yr, "efr_palmgren"] if yr in palm_sub.index else 0
        
        efr_comb = W_PECK * ep + W_COFFIN * ec + W_PALMGREN * epm
        
        efr_combined_rows.append({
            "site_id": SITE_ID,
            "scenario": ssp,
            "year": yr,
            "efr_peck": round(float(ep), 6),
            "efr_coffin": round(float(ec), 6),
            "efr_palmgren": round(float(epm), 6),
            "efr_combined": round(float(efr_comb), 6),
            "mode_peck": "A",
            "mode_coffin": "B",
        })

efr_annual_df = pd.DataFrame(efr_combined_rows)

# IUL computation (Option A: simple scaling)
iul_results = {}
for ssp in SCENARIOS:
    sub = efr_annual_df[efr_annual_df["scenario"] == ssp]
    avg_efr = sub["efr_combined"].mean()
    iul = EUL * (1 - avg_efr)
    years_lost = EUL - iul
    iul_results[ssp] = {"avg_efr": avg_efr, "iul": iul, "years_lost": years_lost}

# Save EFR Parquet
efr_annual_df.to_parquet(EFR_OUTPUT / "efr_annual.parquet", index=False)
print(f"EFR saved: {EFR_OUTPUT / 'efr_annual.parquet'}")
print(f"  {len(efr_annual_df)} rows ({len(SCENARIOS)} scenarios × {n_fut_years} years)")

# Save EFR JSON
efr_json = {
    "site_id": SITE_ID,
    "generated": datetime.now().isoformat(),
    "baseline_years": list(BASELINE_YEARS),
    "future_years": list(FUTURE_YEARS),
    "n_models": len(baseline_counts),
    "models": {
        "peck": {"mode": "A", "input_vars": ["tas", "hurs"], "note": "Linearised 10°C doubles"},
        "coffin_manson": {"mode": "B", "input_vars": ["tasmax", "tasmin"],
                          "note": "Direct freeze-thaw cycle counts from daily data"},
        "palmgren_miner": {"mode": "A", "input_vars": ["sfcWind"],
                           "note": "SCVR≈0 at this site → EFR≈0"},
    },
    "parameters": {
        "T_ref_C": T_REF_C, "RH_ref_pct": RH_REF,
        "Ea_eV": 0.7, "humidity_exponent_n": 2.66,
        "beta_coffin_manson": BETA_CM,
        "std_degrad_rate": STD_DEGRAD,
        "weights": {"peck": W_PECK, "coffin_manson": W_COFFIN, "palmgren_miner": W_PALMGREN},
        "eul_years": EUL,
    },
    "epoch_summary": {},
}

for ssp in SCENARIOS:
    sub = efr_annual_df[efr_annual_df["scenario"] == ssp]
    iul_info = iul_results[ssp]
    efr_json["epoch_summary"][ssp] = {
        "efr_peck_mean": round(float(sub["efr_peck"].mean()), 6),
        "efr_coffin_mean": round(float(sub["efr_coffin"].mean()), 6),
        "efr_palmgren_mean": round(float(sub["efr_palmgren"].mean()), 6),
        "efr_combined_mean": round(float(sub["efr_combined"].mean()), 6),
        "iul_years": round(float(iul_info["iul"]), 1),
        "years_lost": round(float(iul_info["years_lost"]), 1),
    }

json.dump(efr_json, open(EFR_OUTPUT / "efr_results.json", "w"), indent=2)
print(f"EFR JSON saved: {EFR_OUTPUT / 'efr_results.json'}")

# Summary
print("\n── EFR Epoch Summary ──")
print(f"  Weights: Peck's {W_PECK:.0%}, Coffin-Manson {W_COFFIN:.0%}, Palmgren-Miner {W_PALMGREN:.0%}")
for ssp in SCENARIOS:
    es = efr_json["epoch_summary"][ssp]
    print(f"\n  {ssp}:")
    print(f"    Peck's (Mode A):        {es['efr_peck_mean']:+.4f} ({es['efr_peck_mean']*100:+.1f}%)")
    print(f"    Coffin-Manson (Mode B):  {es['efr_coffin_mean']:+.4f} ({es['efr_coffin_mean']*100:+.1f}%)")
    print(f"    Palmgren-Miner (Mode A): {es['efr_palmgren_mean']:+.4f}")
    print(f"    Combined:                {es['efr_combined_mean']:+.4f} ({es['efr_combined_mean']*100:+.1f}%)")
    print(f"    IUL: {es['iul_years']:.1f} years (EUL={EUL}, lost={es['years_lost']:.1f} years)")

In [ ]:
# ── Visualization: HCR Timeline (BI Events Only) ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

colors = {"heat_wave": "#e74c3c", "extreme_precip": "#3498db", "flood_rx5day": "#2980b9",
          "wind_extreme": "#95a5a6", "icing_shutdown": "#1abc9c"}

for ax, ssp in zip(axes, SCENARIOS):
    for hazard in sorted(BI_HAZARDS):
        sub = hcr_annual_df[(hcr_annual_df["hazard"] == hazard) & (hcr_annual_df["scenario"] == ssp)]
        if not sub.empty:
            ax.plot(sub["year"], sub["hcr"] * 100, label=hazard, color=colors.get(hazard, "gray"), linewidth=1.5)
    
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(f"HCR — Channel 1 BI Events ({ssp.upper()})")
    ax.set_xlabel("Year")
    ax.set_ylabel("HCR (%)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(HCR_OUTPUT / "hcr_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: hcr_timeline.png")

In [ ]:
# ── Visualization: EFR Timeline (Channel 2 Degradation) ──────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, ssp in zip(axes, SCENARIOS):
    sub = efr_annual_df[efr_annual_df["scenario"] == ssp]
    
    ax.plot(sub["year"], sub["efr_peck"] * 100, label="Peck's (Mode A)", 
            color="#e74c3c", linewidth=2)
    ax.plot(sub["year"], sub["efr_coffin"] * 100, label="Coffin-Manson (Mode B)", 
            color="#3498db", linewidth=2)
    ax.plot(sub["year"], sub["efr_combined"] * 100, label="Combined", 
            color="#2c3e50", linewidth=2.5, linestyle="--")
    
    # Mark IUL
    iul = iul_results[ssp]["iul"]
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    
    ax.set_title(f"EFR — Channel 2 Degradation ({ssp.upper()})\nIUL = {iul:.1f} years")
    ax.set_xlabel("Year")
    ax.set_ylabel("EFR (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EFR_OUTPUT / "efr_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: efr_timeline.png")

In [ ]:
# ── Output Verification ──────────────────────────────────────────────────────

checks_passed = 0
checks_total = 0

def check(name, condition):
    global checks_passed, checks_total
    checks_total += 1
    status = "✓" if condition else "✗"
    if condition:
        checks_passed += 1
    print(f"  {status} {name}")
    return condition

print("── HCR Checks ──")

# Read back parquet
hcr_read = pd.read_parquet(HCR_OUTPUT / "hcr_annual.parquet")
check("HCR parquet round-trips", len(hcr_read) == len(hcr_annual_df))
check("HCR has exactly 5 BI hazards", len(hcr_read["hazard"].unique()) == 5)
check("freeze_thaw NOT in HCR output", "freeze_thaw" not in hcr_read["hazard"].values)
check("frost_days NOT in HCR output", "frost_days" not in hcr_read["hazard"].values)
check("cold_wave NOT in HCR output", "cold_wave" not in hcr_read["hazard"].values)
check("fire_weather NOT in HCR output", "fire_weather" not in hcr_read["hazard"].values)
check("dry_spell NOT in HCR output", "dry_spell" not in hcr_read["hazard"].values)

# Heat wave sanity
hw_ssp585 = hcr_read[(hcr_read["hazard"] == "heat_wave") & (hcr_read["scenario"] == "ssp585")]
if not hw_ssp585.empty:
    check("heat_wave HCR is positive (SSP585)", hw_ssp585["hcr"].mean() > 0)
    check("heat_wave HCR < 100% (reasonable)", hw_ssp585["hcr"].mean() < 1.0)

# JSON
hcr_j = json.load(open(HCR_OUTPUT / "hcr_results.json"))
check("HCR JSON has risk_indicators", "risk_indicators" in hcr_j)
check("HCR JSON has degradation_hazards_moved_to_efr", "degradation_hazards_moved_to_efr" in hcr_j)

print("\n── EFR Checks ──")

efr_read = pd.read_parquet(EFR_OUTPUT / "efr_annual.parquet")
check("EFR parquet round-trips", len(efr_read) == len(efr_annual_df))

efr_ssp585 = efr_read[efr_read["scenario"] == "ssp585"]
check("Peck's EFR is positive (warming accelerates aging)", efr_ssp585["efr_peck"].mean() > 0)
check("Coffin-Manson EFR is NEGATIVE (fewer freeze-thaw cycles)", efr_ssp585["efr_coffin"].mean() < 0)
check("Combined EFR is positive (Peck's dominates)", efr_ssp585["efr_combined"].mean() > 0)
check("Combined < Peck's alone (C-M partially offsets)", 
      efr_ssp585["efr_combined"].mean() < efr_ssp585["efr_peck"].mean())

efr_j = json.load(open(EFR_OUTPUT / "efr_results.json"))
iul_585 = efr_j["epoch_summary"]["ssp585"]["iul_years"]
check(f"IUL (SSP585) between 20-25 years (got {iul_585:.1f})", 20 <= iul_585 <= 25)
check("Coffin-Manson mode is B", efr_j["models"]["coffin_manson"]["mode"] == "B")
check("Peck's mode is A", efr_j["models"]["peck"]["mode"] == "A")

print(f"\n── Result: {checks_passed}/{checks_total} checks passed ──")

# File listing
print("\nOutput files:")
for f in sorted(HCR_OUTPUT.glob("*")):
    print(f"  {f.name:30s}  {f.stat().st_size:>8,} bytes")
for f in sorted(EFR_OUTPUT.glob("*")):
    print(f"  {f.name:30s}  {f.stat().st_size:>8,} bytes")

## Summary

### What This Notebook Computed

**Channel 1 (HCR — Business Interruption):**
- 5 BI hazards × 2 scenarios × 30 years = 300 annual HCR values
- Pathway A (SCVR × scaling) for heat wave and wind
- Pathway B (direct daily counting) for precipitation and icing
- Risk indicators (fire weather, dry spell) flagged with direction but no $ formula

**Channel 2 (EFR — Equipment Degradation):**
- Peck's thermal aging (Mode A): SCVR_tas → Arrhenius acceleration → EFR_peck
- Coffin-Manson cycling (Mode B): direct freeze-thaw counts → EFR_coffin (NEGATIVE at Hayhurst)
- Palmgren-Miner wind fatigue (Mode A): SCVR_sfcWind ≈ 0 → EFR_palmgren ≈ 0
- Combined EFR + IUL (Impaired Useful Life) estimate

### Key Findings
- **Channel 2 dominates Channel 1** at this site (equipment degradation >> business interruption)
- **Coffin-Manson is negative** — warming reduces freeze-thaw cycles, partially offsetting Peck's aging
- **Heat wave HCR ≈ +17-20%** — validated by Pathway B cross-check
- **Precipitation HCR** correctly captured via Pathway B (Pathway A gives wrong sign)

### Next Steps
- Add HCR + EFR sections to the Streamlit dashboard
- NB05: Financial integration (CFADS/NAV computation)
- Validate EFR against InfraSure operational degradation data